# 02 Train Set EDA

Stage 5 explores only training participants and valid training epochs. The goal is to understand class balance, participant variability, signal summaries, representative raw epochs, and label transitions without using validation or test data for preprocessing or modeling decisions.

This notebook expects the Stage 3 split at `data/interim/split_assignments.csv`. It loads the Stage 4 epoch index from `data/interim/epoch_index.csv` when available, or builds it from raw DREAMT CSVs under `data/raw/` when it is missing. Raw CSVs are also needed for raw signal summaries and representative epoch plots.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
try:
    from IPython.display import Markdown, display
except ModuleNotFoundError:
    class Markdown(str):
        pass

    def display(value):
        print(value)

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data import (
    DEFAULT_EPOCH_INDEX_PATH,
    DEFAULT_RAW_DATA_DIR,
    DEFAULT_SPLIT_ASSIGNMENTS_PATH,
    EXPECTED_SIGNAL_COLUMNS,
    build_epoch_index,
    load_participant_csv,
    load_split_assignments,
)
from src.plots import (
    ACC_MAG_COLUMN,
    add_acc_magnitude,
    collect_epoch_signal_summaries,
    plot_class_balance,
    plot_hypnogram,
    plot_participant_class_distribution,
    plot_raw_epoch_channels,
    plot_signal_summary_by_stage,
    plot_transition_matrix,
    select_representative_epochs,
    summarize_class_balance,
    summarize_participant_class_distribution,
    transition_matrices,
    valid_training_epochs,
)
from src.preprocessing import TARGET_SLEEP_STAGE_LABELS, summarize_epoch_index

try:
    import seaborn as sns

    sns.set_theme(style="whitegrid", context="notebook")
except ModuleNotFoundError:
    plt.style.use("default")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)

raw_data_dir = repo_root / DEFAULT_RAW_DATA_DIR
epoch_index_path = repo_root / DEFAULT_EPOCH_INDEX_PATH
split_assignments_path = repo_root / DEFAULT_SPLIT_ASSIGNMENTS_PATH
figures_dir = repo_root / "results" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
MAX_SIGNAL_SUMMARY_EPOCHS_PER_STAGE = 750


## Load Training Epochs

The split assignment file defines which participants are in training. If the Stage 4 epoch index is missing, this notebook builds it from the local raw DREAMT CSVs before filtering to `split == "train"` for predictive EDA. The cell also summarizes the full epoch index and checks that no non-training participants leak into the training-only EDA. Validation and test rows are left unused.

In [ ]:
split_df = load_split_assignments(split_assignments_path)
train_participants = set(split_df.loc[split_df["split"] == "train", "participant_id"])

if epoch_index_path.exists():
    epoch_index = pd.read_csv(epoch_index_path, dtype={"participant_id": str})
    print(
        f"Loaded existing Stage 4 epoch index from {epoch_index_path}."
    )
else:
    print(
        f"Missing {epoch_index_path}. Building Stage 4 epoch index from "
        f"{raw_data_dir}."
    )
    epoch_index = build_epoch_index(
        raw_dir=raw_data_dir,
        split_assignments_path=split_assignments_path,
        output_path=epoch_index_path,
    )

epoch_index_summary = summarize_epoch_index(epoch_index)
valid_train_epochs = valid_training_epochs(epoch_index)
leaked_participants = set(valid_train_epochs["participant_id"]) - train_participants
if leaked_participants:
    raise ValueError(
        "Training EDA contains participant(s) not assigned to train: "
        f"{sorted(leaked_participants)}"
    )
print(
    f"Using {len(epoch_index):,} indexed epochs; "
    f"{len(valid_train_epochs):,} valid training epochs from "
    f"{valid_train_epochs['participant_id'].nunique():,} participants."
)
display(epoch_index_summary)

split_df["split"].value_counts().rename_axis("split").reset_index(name="participants")


## Class Balance

Class imbalance in the training set informs both modeling and evaluation. This section counts valid training epochs by mapped label, plots the class proportions, and prints a short interpretation. If one class is rare, accuracy can look good while minority-class recall is poor, so later model reports should emphasize macro F1, per-class precision, recall, F1, and confusion matrices.

In [ ]:
if valid_train_epochs.empty:
    class_balance = pd.DataFrame(columns=["mapped_label", "n_epochs", "percentage"])
    print("No valid training epochs are available yet.")
else:
    class_balance = summarize_class_balance(valid_train_epochs)
    display(class_balance)
    plot_class_balance(
        class_balance,
        save_path=figures_dir / "stage5_train_class_balance.png",
    )
    plt.show()

if not class_balance.empty:
    minority = class_balance.sort_values("percentage").iloc[0]
    majority = class_balance.sort_values("percentage").iloc[-1]
    display(
        Markdown(
            f"**Interpretation.** The least represented training class is "
            f"`{minority['mapped_label']}` ({minority['percentage']:.1f}%), "
            f"while `{majority['mapped_label']}` is most represented "
            f"({majority['percentage']:.1f}%). This supports reporting "
            "macro F1 and per-class precision/recall/F1 rather than accuracy alone."
        )
    )


## Participant-Level Class Distributions

Participant-level percentages show whether class balance is driven by a few participants or is broadly shared. The outputs show the smallest and largest participant epoch counts, a participant-by-class distribution plot, and a count of participants with zero epochs for each class. Large heterogeneity would support participant-aware evaluation, class weighting, and caution when interpreting aggregate metrics.

In [ ]:
if valid_train_epochs.empty:
    participant_class_distribution = pd.DataFrame()
    print("No valid training epochs are available yet.")
else:
    participant_class_distribution = summarize_participant_class_distribution(
        valid_train_epochs
    )
    display(participant_class_distribution.sort_values("total_epochs").head(10))
    display(participant_class_distribution.sort_values("total_epochs").tail(10))
    plot_participant_class_distribution(
        participant_class_distribution,
        save_path=figures_dir / "stage5_train_participant_class_distribution.png",
    )
    plt.show()

    zero_class_counts = {
        label: int((participant_class_distribution[f"{label}_count"] == 0).sum())
        for label in TARGET_SLEEP_STAGE_LABELS
    }
    display(
        Markdown(
            "**Interpretation.** Training participants with zero epochs by class: "
            + ", ".join(f"`{label}` = {count}" for label, count in zero_class_counts.items())
            + ". Participants with missing classes can make per-participant behavior uneven, "
            "so participant-level splits remain important."
        )
    )


## Signal Distributions By Sleep Stage

The next analysis uses raw training CSVs to compute simple epoch-level summaries. The first cell samples valid training epochs and processes participant files one at a time; the second reshapes those summaries for descriptive tables and plots. The cap below keeps exploratory plotting responsive; increase it locally if you want full-training summary plots.

In [ ]:
if valid_train_epochs.empty:
    epoch_signal_summaries = pd.DataFrame()
    print("No valid training epochs are available yet.")
elif not raw_data_dir.exists():
    epoch_signal_summaries = pd.DataFrame()
    print(f"Missing raw data directory: {raw_data_dir}")
else:
    try:
        epoch_signal_summaries = collect_epoch_signal_summaries(
            valid_train_epochs,
            raw_data_dir=raw_data_dir,
            max_epochs_per_stage=MAX_SIGNAL_SUMMARY_EPOCHS_PER_STAGE,
            random_state=RANDOM_STATE,
        )
        print(f"Computed signal summaries for {len(epoch_signal_summaries):,} epochs.")
    except FileNotFoundError as exc:
        epoch_signal_summaries = pd.DataFrame()
        print(exc)

epoch_signal_summaries.head()


In [ ]:
summary_signals = ["HR", "IBI", "EDA", "TEMP", ACC_MAG_COLUMN, "BVP"]
summary_stat = "mean"

if epoch_signal_summaries.empty:
    signal_summary_long = pd.DataFrame()
    print("Raw signal summaries are unavailable until local raw CSVs are present.")
else:
    value_columns = [
        f"{signal}_{summary_stat}"
        for signal in summary_signals
        if f"{signal}_{summary_stat}" in epoch_signal_summaries.columns
    ]
    signal_summary_long = epoch_signal_summaries.melt(
        id_vars=["participant_id", "epoch_id", "mapped_label"],
        value_vars=value_columns,
        var_name="signal_stat",
        value_name=summary_stat,
    )
    signal_summary_long["signal"] = signal_summary_long["signal_stat"].str.replace(
        f"_{summary_stat}$",
        "",
        regex=True,
    )
    display(signal_summary_long.groupby(["signal", "mapped_label"])[summary_stat].describe())
    for signal in summary_signals:
        signal_df = signal_summary_long[signal_summary_long["signal"] == signal]
        if signal_df.empty:
            continue
        plot_signal_summary_by_stage(
            signal_df,
            value_col=summary_stat,
            save_path=figures_dir / f"stage5_train_{signal.lower()}_mean_by_stage.png",
        )
        plt.show()

    display(
        Markdown(
            "**Interpretation.** These boxplots are descriptive training-set checks, "
            "not feature-selection proof. Visible stage separation in HR, IBI, EDA, TEMP, "
            "ACC magnitude, or BVP summaries can motivate baseline engineered features, "
            "while overlap supports evaluating nonlinear and temporal models."
        )
    )


## Representative Raw Epochs

A small deterministic sample of raw epochs gives qualitative intuition about channel behavior. The cell samples three training participants, selects one valid epoch per available stage for each sampled participant, reloads the corresponding raw rows, derives acceleration magnitude when possible, and plots the requested channels. These examples are not used to tune models or choose participants, and the section reports when valid epochs or raw CSVs are unavailable.

In [ ]:
representative_channels = ["BVP", ACC_MAG_COLUMN, "EDA", "TEMP", "HR", "IBI"]

if valid_train_epochs.empty or not raw_data_dir.exists():
    representative_epochs = pd.DataFrame()
    print("Representative raw epoch plots require valid training epochs and local raw CSVs.")
else:
    sampled_participant_ids = (
        valid_train_epochs["participant_id"]
        .drop_duplicates()
        .sample(n=min(3, valid_train_epochs["participant_id"].nunique()), random_state=RANDOM_STATE)
        .sort_values()
    )
    sampled_epochs = []
    for participant_id in sampled_participant_ids:
        participant_epochs = valid_train_epochs[valid_train_epochs["participant_id"] == participant_id]
        for label in TARGET_SLEEP_STAGE_LABELS:
            stage_epochs = participant_epochs[participant_epochs["mapped_label"] == label]
            if stage_epochs.empty:
                continue
            sampled_epochs.append(stage_epochs.sample(n=1, random_state=RANDOM_STATE))
    representative_epochs = pd.concat(sampled_epochs).sort_values(
        ["participant_id", "mapped_label", "epoch_id"]
    )
    for _, epoch_row in representative_epochs.iterrows():
        participant_id = epoch_row["participant_id"]
        raw_path = raw_data_dir / f"{participant_id}_whole_df.csv"
        if not raw_path.exists():
            print(f"Skipping {participant_id}; missing {raw_path}")
            continue
        raw_df = load_participant_csv(raw_path, usecols=EXPECTED_SIGNAL_COLUMNS)
        raw_epoch = raw_df.iloc[int(epoch_row["start_row"]): int(epoch_row["end_row"])]
        if {"ACC_X", "ACC_Y", "ACC_Z"}.issubset(raw_epoch.columns):
            raw_epoch = add_acc_magnitude(raw_epoch)
        title = (
            f"{epoch_row['mapped_label']} example: "
            f"{participant_id}, epoch {int(epoch_row['epoch_id'])}"
        )
        save_path = figures_dir / (
            "stage5_representative_epoch_"
            f"{epoch_row['mapped_label'].replace('-', '_')}_{participant_id}.png"
        )
        fig, axes = plot_raw_epoch_channels(
            raw_epoch,
            representative_channels,
            title=title,
        )
        tick_seconds = [0, 5, 10, 15, 20, 25, 30]
        tick_positions = [
            second * (len(raw_epoch) - 1) / 30 for second in tick_seconds
        ]
        axes[-1].set_xticks(tick_positions)
        axes[-1].set_xticklabels(tick_seconds)
        axes[-1].set_xlabel("Seconds within 30-second epoch")
        fig.savefig(save_path, bbox_inches="tight", dpi=150)
        plt.show()


## Temporal Structure

Transitions are computed within each training participant only. The section outputs both transition counts and transition probabilities, then reports the average same-stage probability as a simple temporal-dependence check. The code requires consecutive epoch IDs, so it does not create transitions across participant boundaries or gaps left by excluded epochs.

In [ ]:
if valid_train_epochs.empty:
    transition_counts = pd.DataFrame()
    transition_probabilities = pd.DataFrame()
    print("No valid training epochs are available yet.")
else:
    transition_counts, transition_probabilities = transition_matrices(valid_train_epochs)
    display(transition_counts)
    plot_transition_matrix(
        transition_counts,
        title="Training Sleep-Stage Transition Counts",
        fmt="d",
        save_path=figures_dir / "stage5_train_transition_counts.png",
    )
    plt.show()

    display(transition_probabilities)
    plot_transition_matrix(
        transition_probabilities,
        title="Training Sleep-Stage Transition Probabilities",
        fmt=".2f",
        save_path=figures_dir / "stage5_train_transition_probabilities.png",
    )
    plt.show()

    stay_probability = transition_probabilities.values.diagonal().mean()
    display(
        Markdown(
            f"**Interpretation.** The mean same-stage transition probability is "
            f"{stay_probability:.2f}. Strong diagonal probabilities would support "
            "models with temporal context, such as a context-window CNN or CNN-GRU."
        )
    )


## Optional Hypnogram-Like Plots

A few training participants are plotted to show the temporal organization of labels across valid epochs. The cell chooses the training participants with the most valid epochs so the plots are likely to contain enough sequence structure to inspect. These are descriptive training-only plots and should not be used to revise validation or test evaluation.

In [ ]:
if valid_train_epochs.empty:
    print("No valid training epochs are available yet.")
else:
    hypnogram_participants = (
        valid_train_epochs.groupby("participant_id")
        .size()
        .sort_values(ascending=False)
        .head(3)
        .index
    )
    for participant_id in hypnogram_participants:
        plot_hypnogram(
            valid_train_epochs,
            participant_id=participant_id,
            save_path=figures_dir / f"stage5_hypnogram_{participant_id}.png",
        )
        plt.show()


## Stage 5 Takeaways

Use the executed tables and figures above to answer the Stage 5 questions:

- Is the training data class-imbalanced, and which class is rare?
- Do participant-level class distributions vary enough to affect evaluation?
- Which raw signal summaries show visible training-set differences by sleep stage?
- Do within-participant label transitions show enough temporal dependence to motivate temporal context models?

Downstream preprocessing choices that learn statistics, thresholds, imputers, scalers, feature selectors, or class weights should be fit on the training set only. Validation and test sets should remain untouched until model selection and final evaluation respectively.